#### Installing weight and biases library

In [1]:
!pip install wandb

## Loading the dataset: Used Car Price Prediction

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import wandb
import os

In [3]:
cars_df = pd.read_csv( "https://drive.google.com/uc?export=download&id=10ABViLN4Q7vgIlLvepCduU4B3C6BneJR" )

In [4]:
cars_df.head(5)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,age,KM_Driven,make,mileage,engine,power
0,Ahmedabad,Petrol,Manual,First,5.0,3.90,5,53,toyota,17.71,1197,78.90
1,Coimbatore,Petrol,Manual,First,5.0,3.91,2,38,maruti,15.10,1196,73.00
2,Bangalore,Petrol,Manual,First,4.0,2.15,5,25,tata,25.40,624,37.50
3,Coimbatore,Petrol,Automatic,First,5.0,6.55,3,39,nissan,19.15,1198,75.94
4,Mumbai,Diesel,Manual,First,5.0,7.50,3,55,maruti,24.30,1248,88.50


In [5]:
x_columns = ['KM_Driven', 'Fuel_Type', 'age',
             'Transmission', 'Owner_Type', 'Seats',
             'make', 'mileage', 'engine',
             'power', 'Location']
## model of the car is not included in the model

In [6]:
cars_df.shape

(1038, 12)

In [7]:
cars_df = cars_df[x_columns + ['Price']].dropna()

In [8]:
cars_df.shape

(1037, 12)

## Identifying numerical and categorical features

In [9]:
cat_features = ['Fuel_Type',
                'Transmission', 'Owner_Type',
                'make', 'Location']

In [10]:
num_features = list(set(x_columns) - set(cat_features))

## Utility method for preparing the data

- Splitting the dataset
- Encoding Catgorical Variables

In [11]:
X = cars_df[x_columns]
y = cars_df.Price

In [12]:
# Split the dataset into train and test split
x_train, x_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    train_size = 0.8,
                                                    random_state = 100)

### Creating ML Pipeline

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [14]:
ohe_encoder = OneHotEncoder(handle_unknown='ignore')
scaler = StandardScaler()

## Creating the imputer for columns that have missing values
imputed_num_vars = ['Seats']
non_imputed_num_vars = list(set(num_features) - set(imputed_num_vars))
mean_imputer = SimpleImputer(strategy='mean')


## Pipeline for the applying imputation and then scaling
imputed_num_transformer = Pipeline( steps = [
        ('imputation', mean_imputer),
        ('scaler', scaler)])

non_imputed_num_transformer = Pipeline( steps = [('scaler', scaler)])


## Pipeline for OHE encoding the categorical columns
cat_transformer = Pipeline( steps = [('ohencoder', ohe_encoder)])

## The complete pipeline for applying the required transformatinons to the respective columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num_imputed', imputed_num_transformer, imputed_num_vars),
        ('num_not_imputed', non_imputed_num_transformer, non_imputed_num_vars),
        ('catvars', cat_transformer, cat_features)])

## Initilializing Weights and Biases

In [15]:
os.environ["WANDB_API_KEY"] = "0bea1d0fcbb8912086510be34b0601c8b672cb58"

## Baseline Model: Linear Regression

In [16]:
linear_reg = LinearRegression()

linear_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('linear_model', linear_reg)])
## Pipeline for the applying imputation and then scaling

linear_model.fit(x_train, y_train)

wandb.init(project='mlops_usedcar', config=None, tags = ['Linear Model', 'baseline', 'OHE Encoding'])
wandb.run.name = "LinearModel"
rmse = np.sqrt(mean_squared_error(y_test, linear_model.predict(x_test)))
r2 = linear_model.score(x_test, y_test)

wandb.log( {"rmse" : rmse,
            "r2": r2} )

wandb.Artifact("LinearModel",
               type = 'model',
               description = None)

wandb.save()
wandb.finish()

wandb: Currently logged in as: atharva-programs (atharva-programs-indian-institute-of-management-bangalore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


wandb: WARNING Calling wandb.run.save without any arguments is deprecated.Changes to attributes are automatically persisted.


r2,▁
rmse,▁
r2,0.78906
rmse,0.93277


In [17]:
params = {"max_depth": 10}

dtree = DecisionTreeRegressor(**params)

dtree_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('dt_model', dtree)])


dtree_model.fit(x_train, y_train)

wandb.init(project='mlops_usedcar', config=params, tags = ['Decision Tree',
                                                           'OHE Encoding'])
wandb.run.name = "DecisionTree"
rmse = np.sqrt(mean_squared_error(y_test, dtree_model.predict(x_test)))
r2 = dtree_model.score(x_test, y_test)

wandb.log( {"rmse" : rmse,
            "r2": r2} )

wandb.Artifact("DecisionTree",
               type = 'model',
               description = params)

wandb.save()
wandb.finish()

r2,▁
rmse,▁
r2,0.64687
rmse,1.20688


## Manual Grid Search

In [18]:
from sklearn.model_selection import GridSearchCV

In [19]:
params = { "dt_model__max_depth" : range(5, 10)}

In [20]:
dtree = DecisionTreeRegressor()

dtree_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('dt_model', dtree)])

In [21]:
dt_grid = GridSearchCV(dtree_model,
                       param_grid = params,
                       cv = 10,
                       scoring = 'r2')

In [22]:
dt_grid.fit(x_train, y_train)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num_imputed',
                                                                         Pipeline(steps=[('imputation',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Seats']),
                                                                        ('num_not_imputed',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['power',
                                                                          'engine',
                                                                          'age',
                                                                          'KM_Driven',
                                                                          'mileage']),
                                                                        ('catvars',
                                                                         Pipeline(steps=[('ohencoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Fuel_Type',
                                                                          'Transmission',
                                                                          'Owner_Type',
                                                                          'make',
                                                                          'Location'])])),
                                       ('dt_model', DecisionTreeRegressor())]),
             param_grid={'dt_model__max_depth': range(5, 10)}, scoring='r2')

In [23]:
dt_grid.best_params_

{'dt_model__max_depth': 8}

In [24]:
dt_grid.best_score_

0.7147827516286978

In [25]:
# If dataset is small 10K or so then std deviation should be less than 0.01. Here min std dev is 0.04, more experiment should be done
pd.DataFrame(dt_grid.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_dt_model__max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.027668,0.004285,0.014852,0.002011,5,{'dt_model__max_depth': 5},0.702526,0.562054,0.657594,0.695006,0.738913,0.697023,0.686102,0.737424,0.711134,0.784153,0.697193,0.055769,5
1,0.026414,0.001710,0.014314,0.001259,6,{'dt_model__max_depth': 6},0.714382,0.589445,0.668475,0.726340,0.725160,0.666815,0.713247,0.777868,0.700983,0.703069,0.698578,0.046990,4
2,0.029067,0.001840,0.015201,0.001785,7,{'dt_model__max_depth': 7},0.685606,0.641972,0.723700,0.722573,0.742883,0.631847,0.708625,0.801899,0.738123,0.714811,0.711204,0.046909,2
3,0.034838,0.004209,0.016017,0.002073,8,{'dt_model__max_depth': 8},0.699860,0.667214,0.773073,0.706717,0.706627,0.463273,0.749618,0.837182,0.774946,0.769317,0.714783,0.095948,1
4,0.022816,0.001720,0.009852,0.000564,9,{'dt_model__max_depth': 9},0.710167,0.548070,0.724342,0.702577,0.699019,0.600928,0.740261,0.846471,0.754769,0.720030,0.704664,0.077369,3


### Using Sweep Features

In [26]:
def train_decision_tree(config=None):
    # Initialize WandB
    with wandb.init(config=config):
        config = wandb.config

        dtree = DecisionTreeRegressor(max_depth=config.max_depth)

        dtree_model = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('dt_model', dtree)])
        dtree_model.fit(x_train, y_train)

        # Evaluate the model
        rmse = np.sqrt(mean_squared_error(y_test, dtree_model.predict(x_test)))
        r2 = dtree_model.score(x_test, y_test)

        # Log metrics to WandB
        wandb.log( {"rmse" : rmse,
                    "r2": r2,
                    "max_depth": config.max_depth} )


In [27]:
sweep_config = {
    "method": "grid",  # Can be 'grid', 'random', or 'bayes'
    "metric": {"name": "r2", "goal": "maximize"},
    "parameters": {
        "max_depth": {
            "values": [4, 6, 8, 12]  # Depths to evaluate
        },
    },
}

In [28]:
sweep_id = wandb.sweep(sweep_config, project="mlops_usedcar")

Create sweep with ID: 324h6lfr
Sweep URL: https://wandb.ai/atharva-programs-indian-institute-of-management-bangalore/mlops_usedcar/sweeps/324h6lfr


In [29]:
wandb.agent(sweep_id,
            function=train_decision_tree)  # Run all experiments

wandb: Agent Starting Run: ij62nc95 with config:
wandb: 	max_depth: 4


max_depth,▁
r2,▁
rmse,▁
max_depth,4
r2,0.66376
rmse,1.17766


wandb: Agent Starting Run: 31r42fuw with config:
wandb: 	max_depth: 6


max_depth,▁
r2,▁
rmse,▁
max_depth,6
r2,0.74753
rmse,1.02046


wandb: Agent Starting Run: dkz6ewwi with config:
wandb: 	max_depth: 8


max_depth,▁
r2,▁
rmse,▁
max_depth,8
r2,0.75908
rmse,0.99686


wandb: Agent Starting Run: vdcc8o2s with config:
wandb: 	max_depth: 12


max_depth,▁
r2,▁
rmse,▁
max_depth,12
r2,0.65321
rmse,1.19599


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


## Get Experiment Details

In [32]:
api = wandb.Api()

all_runs = api.runs("atharva-programs-indian-institute-of-management-bangalore/mlops_usedcar", order="+summary_metrics.rmse")

for run in all_runs:
  print(f"Model Name: {run.name} and R2 {run.summary.get('r2')}")
  print(run.config)

Model Name: LinearModel and R2 0.7890612721548755
{}
Model Name: snowy-sweep-3 and R2 0.7590762095646219
{'max_depth': 8}
Model Name: cosmic-sweep-2 and R2 0.7475343291441993
{'max_depth': 6}
Model Name: lunar-sweep-1 and R2 0.6637596140757374
{'max_depth': 4}
Model Name: light-sweep-4 and R2 0.6532094724955178
{'max_depth': 12}
Model Name: DecisionTree and R2 0.6468692270582239
{'max_depth': 10}


### Storing the model into a file

In [33]:
from joblib import dump

MODEL_DIR = "./carsmodel"

os.mkdir(MODEL_DIR)
dump(linear_model, MODEL_DIR + "/" + 'cars.pkl')

['./carsmodel/cars.pkl']

### Logging the model artifact in the tracking tools (weights and Biases)

In [34]:
wandb.init(project='mlops_usedcar',
           config=None,
           tags = ['Final Model'])
wandb.run.name = "FinalModel"

In [35]:
model_artifact = wandb.Artifact("Linear_Model_UsedCar",
                                type = 'model',
                                description = 'Linear Model for used car price prediction')

In [36]:
model_artifact.add_dir(MODEL_DIR)

wandb: Adding directory to artifact (./carsmodel)... Done. 0.0s


In [37]:
wandb.run.log_artifact(model_artifact)

<Artifact Linear_Model_UsedCar>

In [38]:
wandb.save()
wandb.finish()

In [ ]:
import sklearn
sklearn.__version__